# 4. Interpretable Risk Modeling

Compare majority and persistence baselines with Logistic Regression and a depth-4 Decision Tree. Numeric feature order, preprocessing, model selection and stable ranking are shared with Notebook 05 in [analysis_helpers.py](analysis_helpers.py).

Feature years: train 2014–2018, validation 2019–2020, test 2021–2022. Targets refer to the following fiscal year. Train alone supplies clipping bounds, imputation medians, sector vocabulary and scaling. Validation average precision chooses unweighted versus balanced Logistic Regression; an exact tie keeps unweighted. Tree depth 4 and minimum leaf size 50 are fixed in advance. The test set is used only for final evaluation.

Persistence uses the current-year two-of-three rule; a missing current signal contributes no positive evidence. This fallback is explicit and is separate from target construction, which requires all next-year signals.

In [1]:
from pathlib import Path
import sys

# Works from the project root or any folder beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/analysis_helpers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project folder.")
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.tree import export_text
from analysis_helpers import fit_models, read_panel, model_comparison, score_table

bundle = fit_models(read_panel())
for label, (x, y) in zip(("Train 2014–2018", "Validation 2019–2020", "Test 2021–2022"), bundle["parts"]):
    print(f"{label}: {len(y):,} company-years; pressure rate {y.mean():.1%}")
print("Validation average precision (unweighted, balanced):", np.round(bundle["validation_ap"], 6))
print("Selected Logistic Regression class_weight:", bundle["class_weight"])
comparison = model_comparison(bundle)
print("\nTest metrics at threshold 0.5; PR-AUC is average precision:")
print(comparison[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]].round(6).to_string(index=False))
print("\nWithin-year capacity metrics, pooled over test company-years:")
print(comparison[["model", "recall_at_10", "precision_at_10", "recall_at_15", "precision_at_15", "recall_at_20", "precision_at_20"]].round(6).to_string(index=False))
scores = score_table(bundle)
print("\nLogistic confusion matrix: rows actual 0/1; columns predicted 0/1")
print(confusion_matrix(scores.actual_label, scores.predicted_score.ge(0.5)))
coefficients = pd.DataFrame({"feature": bundle["parts"][0][0].columns,
                             "coefficient": bundle["lr"].named_steps["logisticregression"].coef_[0]})
coefficients["odds_ratio"] = np.exp(coefficients.coefficient)
coefficients = coefficients.iloc[coefficients.coefficient.abs().argsort()[::-1]]
print("\nStandardized logistic coefficients and odds ratios:")
print(coefficients.round(4).to_string(index=False))
print("\nShallow tree rules:")
print(export_text(bundle["tree"], feature_names=list(bundle["parts"][0][0].columns), decimals=3))
print("\nAnnual queue counts:")
print(pd.crosstab(scores.feature_year, scores.risk_tier).to_string())
scores.to_csv(ROOT / "data/processed/model_scores.csv", index=False)
print("Saved data/processed/model_scores.csv; run Notebook 05 to add sensitivity scores.")

Train 2014–2018: 7,701 company-years; pressure rate 11.4%
Validation 2019–2020: 3,124 company-years; pressure rate 18.0%
Test 2021–2022: 3,305 company-years; pressure rate 16.9%
Validation average precision (unweighted, balanced): [0.489409 0.458825]
Selected Logistic Regression class_weight: None

Test metrics at threshold 0.5; PR-AUC is average precision:
              model  accuracy  precision   recall       f1  roc_auc   pr_auc
           Majority  0.830560   0.000000 0.000000 0.000000 0.500000 0.169440
        Persistence  0.882300   0.663480 0.619643 0.640813 0.777763 0.475568
Logistic Regression  0.881694   0.836653 0.375000 0.517879 0.883610 0.696397
      Decision Tree  0.885325   0.730280 0.512500 0.602308 0.846687 0.593718

Within-year capacity metrics, pooled over test company-years:
              model  recall_at_10  precision_at_10  recall_at_15  precision_at_15  recall_at_20  precision_at_20
           Majority      0.037500         0.063444      0.058929         0.0663

## Capacity and interpretation

Rank each test feature year by risk score descending, then ten-character CIK ascending, using stable sorting. High contains `ceil(10% × n)` rows; High plus Medium contains `ceil(15% × n)`. Routine contains the remainder. Small rounding differences mean actual coverage can slightly exceed the nominal capacity. All capacity metrics use the same annual rule and pool counts across years.

PR-AUC here means average precision. Threshold metrics use 0.5 for comparability; the review queue uses rank, not that threshold. Repeated tree leaf scores and binary persistence scores make the CIK tie-break material; CIK has no predictive interpretation and is never a model feature.

Logistic coefficients are conditional associations after clipping, imputation and scaling. Numeric odds ratios correspond to one training standard deviation; standardized sector indicators are not direct sector-versus-reference odds ratios. Correlated features make individual coefficients less stable. The risk score is a screening and ranking signal, not a probability of default, credit rating or automatic decision.